In [ ]:
# Colab dependency conflicts require a workaround installing dependencies. Don't ask...
# !pip install "unstructured[pdf]" --no-deps
# !pip install emoji psutil backoff dataclasses-json filetype langdetect python-iso639 python-magic python-oxmsg python-oxmsg rapidfuzz unstructured-client

# # !pip install sentence-transformers # Avoid overriding Colab's CUDA libraries. Remove --no-deps when installing locally.
# !pip install sentence-transformers --no-deps

# !pip install pymupdf
# !pip install langchain-community langchain-text-splitters
# !python -m spacy download en

# !pip install chromadb

In [ ]:
import re
import hashlib
from dateutil.parser import parse
from pprint import pprint
from datetime import datetime

In [ ]:
DEVICE = "cuda"   # "cpu"

In [ ]:
NAIVE_DATE_RE_COMPILED = re.compile(r"[A-Z][a-z]{2,10}\s\d{1,2},\s2024")  # E.g. "December 27, 2024"

## Load a PDF Document

In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader

In [ ]:
loader = PyMuPDFLoader("News Digest 2024 - Question of Cities.pdf")
raw_pdf = loader.load()

In [ ]:
raw_pdf[0].metadata

In [ ]:
pdf_title = raw_pdf[0].metadata["title"]

Things to keep in mind:
1. We don't want any overlap of text between any two articles (obviously). Therefore, we need to treat every article as being it's own document and chunk it individually. It looks like we can use the text on the "Read the full story" link as a separator between articles.
2. An article may split mid-page.
3. It would be nice to have the article publish date as metadata.

In [ ]:
raw_whole_pdf_text = "\n".join([page.page_content for page in raw_pdf])
len(raw_whole_pdf_text)

Now, we need to split by article.

In [ ]:
raw_article_documents_text = raw_whole_pdf_text.split("Read the full story")
len(raw_article_documents_text)

In [ ]:
raw_article_documents_text[5]

Good. Next step is to create a list of dictionaries with each dictionary containing a single article, it's publishing date and the title of the document it originated from. We'll try to extract the publishing date from each article, and assign it to it and the following articles, until we hit a new publishing date.

For convenience reasons, we'll stick with `LangChain`'s `Document` type for the article content and metadata. It will help avoid all kinds of conversions when we use other `LangChain` services.

In [ ]:
article_documents: list[Document] = []
article_date: str = ""

for raw_article_text in raw_article_documents_text:

    # The articles are ordred by date. The date only shows next to the first article from that date.
    new_article_date: re.Match|None = NAIVE_DATE_RE_COMPILED.search(raw_article_text)
    if new_article_date:
        article_date = new_article_date.group()

    article_documents.append(
        Document(
            page_content=raw_article_text,
            metadata={
                "publish-date": article_date,
                "doc-title": pdf_title,
            }
        )
    )

len(article_documents)

In [ ]:
print(article_documents[5])

## Cleanup and Normalize

In [ ]:
from unstructured.cleaners.core import clean, clean_bullets, clean_extra_whitespace

In [ ]:
def clean_and_normalize_document(article_document: Document, article_publish_date: str) -> Document:
    
    text = article_document.page_content.replace("SEARCH\n", "").replace("MENU\n", "").replace("News Digest\n", "")
    
    if article_publish_date:
        text = text.replace(article_publish_date, "")
    
    text = clean(text)
    text = clean_bullets(text)
    text = clean_extra_whitespace(text)
    text = text.strip()

    article_document.page_content = text
    
    return article_document

In [ ]:
for article_document in article_documents:
    article_publish_date = article_document.metadata["publish-date"]
    clean_and_normalize_document(article_document, article_publish_date=article_publish_date)
    if article_publish_date:
        article_document.metadata["publish-date"] = parse(article_publish_date).isoformat()

In [ ]:
print(article_documents[5])

## Chunking

Okay. We have our raw articles cleaned-up, normalized and with metadata added. The next thing to do is split each article into chunks. Each chunk will share the same metadata as the article.

#### Sentence-Based Chunking

In [ ]:
from langchain_text_splitters import SpacyTextSplitter

In [ ]:
SpacyTextSplitter?

In [ ]:
splitter = SpacyTextSplitter(chunk_size=800, chunk_overlap=300,)  

In [ ]:
article_chunks: list[Document] = splitter.split_documents(article_documents)

> **Note**: `SpacyTextSplitter` does not necessarily do sentence overlapping between any two consecutive chunks. It depends on the `chunk_size` and `chunk_overlap` arguments when applied to a given text and varied sentence lengths. Sometimes, the last sentence in chunk N does not fit in chunk overlap character allowance (`chunk_overlap`) in chunk N+1.

In [ ]:
len(article_chunks)

In [ ]:
print(article_chunks[0])
print("~" * 100)
print(article_chunks[1])
print("~" * 100)
print(article_chunks[2])

Seems like we have our overlapping.

## Vectorizing

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [ ]:
CHROMADB_CLIENT = chromadb.Client()

In [ ]:
# Embeddings take around 75 seconds on CPU.
bge_ef = SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5", device=DEVICE)  # device="cuda"

CHROMADB_COLLECTION = CHROMADB_CLIENT.get_or_create_collection(
    name="augmentation-articles-collection-1",
    embedding_function=bge_ef,   # chromadb.utils.embedding_functions.ONNXMiniLM_L6_V2()
    configuration={
        "hnsw": {
            "space": "cosine"
        },
    }
)

In [ ]:
print(f">>> About to ingest {len(article_chunks)} chunks...")

In [ ]:
for chunk_document in article_chunks:
    
    content = chunk_document.page_content
    CHROMADB_COLLECTION.add(
            ids=[hashlib.md5(content.encode()).hexdigest()[-8:]],
            documents=[content],  # Text for embedding
            metadatas=[{
                **chunk_document.metadata,
                "token-count": int(len(content.split()) * 1.4),
            }],
    )

In [ ]:
CHROMADB_COLLECTION.count()

In [ ]:
print(f"Wait a second. Didn't we have {len(article_chunks)} chunks?!")

In [ ]:
texts = [chunk.page_content for chunk in article_chunks]

In [ ]:
len(set(texts))

In [ ]:
for i, text in enumerate(texts):
    if text in texts[i+1:]:
        print(f">>> chunk #{i} same as #{texts.index(text, i+1)}") 

In [ ]:
article_chunks[36].page_content

In [ ]:
article_chunks[49].page_content

Yep...

## Retrieval

In [ ]:
user_query = "Do data centres have the potential to affect climate? Provide examples and numbers."

In [ ]:
search_results = CHROMADB_COLLECTION.query(
    query_texts=[user_query],
    n_results=7,
    include=["documents", "metadatas", "distances"],
)

In [ ]:
pprint(list(zip(search_results["distances"][0], search_results["documents"][0])))

In [ ]:
vector_store_chunk_ids = search_results["ids"][0]
vector_store_chunk_ids

### Re-ranking

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
rerank_model = CrossEncoder("mixedbread-ai/mxbai-rerank-large-v1")

In [ ]:
reranked_chunks = rerank_model.rank(user_query, search_results["documents"][0], return_documents=True, top_k=7)

In [ ]:
reranked_chunks

For this exercise, we'll have the threshold score set to `0.4`.

In [ ]:
THRESHOLD_SCORE = 0.4

In [ ]:
final_augmentation_chunks = [reranked_chunk["text"] for reranked_chunk in reranked_chunks if reranked_chunk["score"] >= THRESHOLD_SCORE]
len(final_augmentation_chunks)

In [ ]:
final_augmentation_chunk_ids = [
    vector_store_chunk_ids[reranked_chunk["corpus_id"]] for reranked_chunk in reranked_chunks if reranked_chunk["score"] >= THRESHOLD_SCORE
]
final_augmentation_chunk_ids

## Prompting

In [ ]:
STUFFING_PROMPT_TEMPLATE = """
You are an expert assistant. Answer the user's question based MOSTLY on the provided context.

If the context does not contain the answer, say "I don't know based on the context."

Context:
{context}

Question: {user_query}

Answer:
"""

In [ ]:
context = "\n---\n".join(final_augmentation_chunks)
print(context)

In [ ]:
final_prompt = STUFFING_PROMPT_TEMPLATE.format(context=context, user_query=user_query)
print(final_prompt)

## Generation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
HF_LLM_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"

In [ ]:
LLM_TOKENIZER = AutoTokenizer.from_pretrained(HF_LLM_MODEL_ID)
LLM_MODEL = AutoModelForCausalLM.from_pretrained(HF_LLM_MODEL_ID, dtype=torch.bfloat16, device_map="auto")

In [ ]:
GENERATOR = pipeline(task="text-generation", model=LLM_MODEL, tokenizer=LLM_TOKENIZER)

In [ ]:
response = GENERATOR(final_prompt, max_new_tokens=500, return_full_text=False, temperature=0.01)
print(response[0]["generated_text"])

<hr/>